# 39. Seven rows are not seven days

**Track:** debug-the-ai | **Difficulty:** medium | **Tags:** window functions, rolling sum, rowsBetween, rangeBetween, missing days

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("039-rolling-seven-days")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "daily_sales": """
UEFSMRUEFYABFVhMFRAVABIAAEAEAgAJAQAFCQcEAAENCAAHDQgACA0IAAMNCDwEAAAAAAAAAAYAAAAAAAAAFQAV3gEV5AEsFZgE
FRAVBhUGHBgICAAAAAAAAAAYCAEAAAAAAAAAFgAoCAgAAAAAAAAAGAgBAAAAAAAAABERAAAAb/BuAwAAAJgEAQNFQCBgY+V1+cVk
oBc08qt81RJAgpJKigsoLhT4P2opzzYO/KUkQhsL771d1/uw/cbUajH92u2plpzU1XD/e/12tjwH2NZDelbRsIsYmaIR6mUoeCg5
hQzsjZ8aqj3qXj/RwXTvKQIAFQQV6AIV8AJMFVoVABIAALQB8LPuTgAA3U4AAPhOAADaTgAA7U4AANdOAAD8TgAA804AANxOAAD6
TgAA+U4AANVOAADjTgAA5U4AANhOAADpTgAA5E4AAPBOAAD2TgAA/k4AAPROAADUTgAA+04AAN9OAADqTgAA704AAOFOAADZTgAA
6E4AANZOAAD9TgAA3k4AAOtOAADiTgAA004AAOxOAAD/TgAA4E4AAPFOAAD1TgAA204AAOdOAAD3TgAA8k4AAOZOAAAVABWqAxWy
AywVmAQVEBUGFQYcGAT/TgAAGATTTgAAFgAoBP9OAAAYBNNOAAAREQAAANUB8NQDAAAAmAQBBkVAIAxEYRxIoiwA0zgPFEnNBFGV
VV0HBmTaxkGT1GydVwwfuIWbAopjxpBAB5ZekwWKcFQGk3gK9hhI8DDmiRrKlSxOQRpOZHJcaWpFd31immUWYKBqoXnI+oWiEC4s
mahpNGQa+6yqOo1SNxbMgWgAAqLVZ0KLl1TT8Kzb9gWRqJQnWSWnNJBIuUIns2WjQCzcEEimkgpemaTmOmrYIXUnaSUpiBbD+qmT
RB0RGn2kFgYlg3XlWZjVIZ6cuIZCRDWh+DXA5B3YgjQAAAAVBBXaFBXkFEwVkgQVABIAAK0K9CwFAAAYmzgAADB4NAAAECZgAAAl
/4oAACD5NgAAGn6eAAA17BoAADAknAAAJ3VeAAAyMQYAACoBUgAAN8mkAAAxvQIAAFEhJAAAN/ZeAAA1pNAAAB7HTAAAEbaSAAAK
1zIAADqD4gAAMyDeAAAb+uQAABfe8AAAL/wuAAAcCSYAAB8iHgAAPxb6AAA4JaIAABXShgAAGMfAAAA6MdoAABpfwgAAHloeAAA3
d5wAAA1y+AAALd+OAAA+NAYAABBw/AAAFSiaAAA+gSwAADEVbgAAODTeAAAnPOwAABqs6AAAEFguAAAeafAAACIlmgAAMoJ4AAAw
0j4AAB0PpgAAKZ7iAAAdvtgAAB2PYgAAVx0sAAA3hOQAACjl7AAAItYqAAAhRo4AACXjzgAARq3AAAAzo4gAADQysgAANmjoAAAa
F7AAAC5fSgAAMa/sAAAZ5eIAACQ+IAAAHViAAAAY9KwAACyIZAAADJLAAAAaC8YAABWuMgAAKCmkAAApAagAAExLDgAANkOaAAA5
PegAABjgXAAAFYhOAAAjiRIAACXJDAAAP2lmAABCDfYAABdB6AAASwzkAAAjqLYAAC+/ogAAGAsUAAAWf8QAACRP5gAAI8GEAAAd
NJAAABGfhgAAIWN2AAAcU/QAADRpYgAAKjAAAABUtfAAABjZIgAAVIDQAABAkLYAACcuFAAAGUpqAABUDGgAABTN+gAAMxGiAAAk
gLoAAEDxMgAADWk0AAA6sDgAABZXiAAAG7CsAAAZJOoAAE8S+AAACsRyAAAUW4YAACb66AAAJTUAAAAvkMIAABZGJgAAQCS0AAAy
oPAAACYjSAAAEj1WAAA6YxIAADDjbgAAMXYcAABAU5QAAD+slgAAM8OQAAAw2kAAADt7igAADqlSAAAQaooAACUmjAAAT/B0AAAT
VmQAAERprAAASx4UAAAuMfoAADPIDgAAR846AABTzO4AACU9NAAAOKz8AAA8fiIAACEanAAAIRbmAAAkSHoAAC5frgAAJY2sAAAp
XzYAACa6dAAAF7tkAABBu1gAABfDygAAFeZAAABEHhYAABFbjgAAQMTcAABNLNYAABOJLAAAMr48AABP9MAAAD/6IAAALCh+AABF
X8QAACL1zgAATr/2AABE1hIAADYKkgAAE3MaAAAc1Q4AACSNCAAAJRT4AAApIX4AAFDiDgAAPZnsAAAWMp4AACamugAAQTJuAAAj
nMwAAA51LAAAIjkiAABCsjgAAFFtggAAPs/iAAAwkNAAAEjXRAAAJCQmAABKynwAAEx+0AAAMFdkAAAbGOoAACN/5AAAVOqsAAAx
wqwAABRFQgAAMHNSAAAqPLIAACNZagAAL75EAAAi5J4AADyDaAAAQvkeAAAMeogAADH7tAAAOQhkAAAy9nwAACAvdAAAHEowAAA2
HSAAABLBLAAAGRHGAAAW7MAAACNFfgAAIQCiAABEPboAAA/JmgAAJVFSAAA3/NAAADW56AAAOGdCAAAgDg4AADBzhAAALY9IAAAX
s5QAAE0MOAAAE7jUAAAoOXYAAAw9ZgAAFgS4AABB2/YAAD0InAAAN4skAAAekfoAAESFmgAAI5DiAABQupoAADkYmgAAMwwqAAAe
7vIAABHCSgAAP/LmAAAiH1oAAEo4AAAAF6eqAAA230QAADK5jAAAJ+8+AAA0EkYAABANkgAAMD2cAABIfZ4AAB2jsgAAEk+AAABP
3uAAADeTigAAFmPWAAAUo2YAACvJYAAAILroAAAfdVIVABX2BBWABSwVmAQVEBUGFQYcGAUAAFcdLBgFAAAKxHIWACgFAABXHSwY
BQAACsRyEREAAAC7AvQ6AQMAAACYBAEJRQACCBhAoICBAwgSKFjAoIGDBxAiSJhAoYKFCxgyaNjAoYOHDyBCiBhBooSJEyhSqFjB
ooWLFzBiyJhBo4aNGzhy6NjBo4ePH0CCCBlCpIiRI0iSKFnCpImTJ1CiSJlCpYqVK1iyaNnCpYuXL2DCiBlDpoyZM2jSqFnDpo2b
N3DiyJlDp46dO3jy6NnDp4+fP4ACCRpEqJChQ4gSKVrEqJGjR5AiSZpEqRIKS5cwZdK0iVMnT59AhRI1ilQpU6dQpVK1ilUrV69g
xZI1i1YtW7dw5dK1i1cvX7+ABRM2jFgxY8eQJVO2bAMzCM2cPYMWTdo0atWsXcOWTds2bt28fQMXTtw4cuXMnUOXTt06du3cvYMX
T948evXs3cOXT98+fv38/QMYUOBAggUNHkQIAAAAABUEGUw1ABgGc2NoZW1hFQYAFQQlAhgIc3RvcmVfaWQAFQIlAhgJc2FsZV9k
YXRlJQxMbAAAABUOFQoVAhgHcmV2ZW51ZSUKFQQVFCxcFQQVFAAAABaYBBkcGTwmABwVBBk1AAYQGRgIc3RvcmVfaWQVAhaYBBaA
BBbeAyZ+JggcGAgIAAAAAAAAABgIAQAAAAAAAAAWACgICAAAAAAAAAAYCAEAAAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkm
AJgEAAAAJgAcFQIZNQAGEBkYCXNhbGVfZGF0ZRUCFpgEFpYHFqYHJvYGJuYDHBgE/04AABgE004AABYAKAT/TgAAGATTTgAAEREA
GSwVBBUAFQIAFQAVEBUCADwpBhkmAJgEAAAAJgAcFQ4ZNQAGEBkYB3JldmVudWUVAhaYBBbeGhbyGiaSICaMCxwYBQAAVx0sGAUA
AArEchYAKAUAAFcdLBgFAAAKxHIREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAmAQAAAAW9CUWmAQmCBb2JQAZHBgMQVJST1c6c2No
ZW1hGNgCLy8vLy8vZ0FBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFB
QU1BQUFDSUFBQUFRQUFBQUFRQUFBQ1UvLy8vQUFBQkJ4QUFBQUFnQUFBQUJBQUFBQUFBQUFBSEFBQUFjbVYyWlc1MVpRQUlBQXdB
QkFBSUFBZ0FBQUFLQUFBQUFnQUFBTXovLy84QUFBRUlFQUFBQUNBQUFBQUVBQUFBQUFBQUFBa0FBQUJ6WVd4bFgyUmhkR1VBQmdB
SUFBWUFCZ0FBQUFBQUFBQVFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFJUUFBQUFKQUFBQUFRQUFBQUFBQUFBQ0FBQUFI
TjBiM0psWDJsa0FBQUFBQWdBREFBSUFBY0FDQUFBQUFBQUFBRkFBQUFBQUFBQUFBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJz
aW9uIDI1LjAuMRk8HAAAHAAAHAAAADADAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "039-rolling-seven-days"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

daily_sales = spark.read.parquet((_data_dir / "daily_sales.parquet").resolve().as_posix())
daily_sales.createOrReplaceTempView("daily_sales")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'revenue': '8adccabd337fd534ac7971bc2ec7960f45d1ae0634169d55d1494b6cdeb3f040',
                         'rolling_7d_revenue': '08a149526fe3bbb8ad93e64135ec73c81bd732e4693e2dd16f2fc7d48b3c9d59',
                         'sale_date': '08870211fc73cbcbbd1bbc2aa00e3037f022533fbdc82005681076f392683c93',
                         'store_id': 'f02da3bfafd13ab892db92d054a659ca10d6af47473a7f63dd9ca95f29592e79'},
 'columns': [['revenue', 'decimal(10,2)'],
             ['rolling_7d_revenue', 'decimal(14,2)'],
             ['sale_date', 'date'],
             ['store_id', 'bigint']],
 'fingerprint': 'cf42862a5e7a195e937f363474c473b028af55a3bd7b0731b224696d988e3470',
 'order_matters': False,
 'row_count': 268}

check = make_check(EXPECTED)

## The situation

Store managers get a rolling 7-day revenue number every morning. An AI assistant
wrote the job in the answer cell. It was tested on the flagship store, which is open
every day, and every number was right.

The manager of a store that opens only at weekends then reported that her "7-day"
revenue was larger than everything the store had sold in the last three weeks.

Find the bug and fix it.

## Input tables

### `daily_sales`

One row per store per day **on which the store sold something**. A day the store was
closed has no row.

| Column | Type | Description |
|---|---|---|
| `store_id` | BIGINT | The store. |
| `sale_date` | DATE | The day. Unique per store. |
| `revenue` | DECIMAL(10,2) | The store's revenue that day. Never NULL. |

## Requirement

Return every row of `daily_sales` with the store's rolling 7-day revenue.

- `rolling_7d_revenue` is the sum of the store's `revenue` over the 7 calendar days
  ending on `sale_date`: from `sale_date` minus 6 days up to and including
  `sale_date`.
- A day without a row adds nothing. It still counts as one of the 7 days.
- No rows are added for the missing days.

Example: a store sold on 1 April, 2 April and 10 April. For 10 April the window is
4 April to 10 April, so the rolling revenue is the revenue of 10 April alone.

## Expected output

One row per row of `daily_sales`.

| Column | Type | Description |
|---|---|---|
| `store_id` | BIGINT | |
| `sale_date` | DATE | |
| `revenue` | DECIMAL(10,2) | |
| `rolling_7d_revenue` | DECIMAL(14,2) | Revenue of the 7 calendar days ending on `sale_date`. |

In [ ]:
daily_sales.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `daily_sales`.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F


def solve(spark):
    """Rolling 7-day revenue per store."""
    sales = spark.table("daily_sales")

    # Window over the current day and the six days before it
    last_7_days = (
        Window.partitionBy("store_id")
        .orderBy("sale_date")
        .rowsBetween(-6, 0)
    )

    result = sales.select(
        "store_id",
        "sale_date",
        "revenue",
        F.sum("revenue").over(last_7_days).cast("decimal(14,2)").alias("rolling_7d_revenue"),
    )

    return result


result = solve(spark)
result.show()

In [ ]:
check(result)